# HW2 - Build a "Silver" flat view - SOLUTION
**Slides 4, 15, 17, 25.** Try [`../HW2_silver_flatten.ipynb`](../HW2_silver_flatten.ipynb) first, then compare.

Two Silver tables in plain Python (dev_env has no pandas): `orders_flat` (one row per order) and `order_items` (one row per item).

**Before you run it:** `bash lab/reset.sh` from the repo folder on your laptop.

In [ ]:
import os
from pymongo import MongoClient

# Inside dev_env the host is "mongo" (Jupyter does not see $MONGO_URI, so we fall back to the same value).
URI = os.getenv("MONGO_URI", "mongodb://mongo:27017/?directConnection=true")
client = MongoClient(URI, serverSelectionTimeoutMS=5000)
db = client["shop"]
coll = db["orders"]
print("connected to", URI, "| orders:", coll.count_documents({}))

docs = list(coll.find())
print("loaded", len(docs), "documents")

## 1. Helpers: one function per messy field
- **email** (slide 15): v1 has `customer.email`, v2 has `customer.contact.email`
- **total** (slide 11): `Decimal128`, `float`, `str` or missing -> a `float`, or `None` when missing
- **status**: lowercase, `None` when missing
- **created_at** (slide 11): BSON Date (`datetime`), ISO string like `"2026-03-31T15:06:00Z"`, or epoch **milliseconds** (`int`) -> a `datetime` in UTC

PyMongo returns BSON Dates as *naive* UTC datetimes. Return **naive UTC** datetimes for all three formats so they compare cleanly.

In [ ]:
from datetime import datetime, timezone
from bson import Decimal128

def unified_email(doc):
    customer = doc["customer"]
    return customer.get("email") or customer.get("contact", {}).get("email")

def total_as_number(doc):
    value = doc.get("total")
    if value is None:
        return None
    if isinstance(value, Decimal128):
        return float(value.to_decimal())
    return float(value)                      # float stays float, "499.00" -> 499.0

def clean_status(doc):
    status = doc.get("status")
    return status.lower() if status is not None else None

def created_at_as_datetime(doc):
    value = doc["created_at"]
    if isinstance(value, datetime):
        return value                          # already naive UTC from PyMongo
    if isinstance(value, str):
        return datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ")
    # epoch milliseconds (int / bson Int64)
    return datetime.fromtimestamp(int(value) / 1000, tz=timezone.utc).replace(tzinfo=None)

## 2. Build `orders_flat` (one row per order)
Each row is a dict with keys: `_id`, `order_id`, `schema_v`, `email`, `total`, `status`, `created_at`, `city`.
Keep `_id` - it is the MERGE key downstream (`order_id` has duplicates!).

In [ ]:
orders_flat = [
    {
        "_id": d["_id"],
        "order_id": d["order_id"],
        "schema_v": d["schema_v"],
        "email": unified_email(d),
        "total": total_as_number(d),
        "status": clean_status(d),
        "created_at": created_at_as_datetime(d),
        "city": d["shipping"]["address"]["city"],
    }
    for d in docs
]

In [ ]:
print(len(orders_flat), "rows")
print(orders_flat[:1])

## 3. Build `order_items` (one row per item, slides 17 and 25)
Each row: `order_id`, `item_idx` (position in the array), `sku`, `qty` (**int** - some are strings like `"2"`), `unit_price`.

**Decide:** what happens to the 40 orders with an empty `items` array?
- `KEEP_EMPTY_ORDERS = True` -> keep one row per empty order with `None` item fields (like `explode_outer`)
- `KEEP_EMPTY_ORDERS = False` -> drop them (like `explode` / `$unwind` without options)

In [ ]:
KEEP_EMPTY_ORDERS = False   # item table = real items only; empty orders still exist in orders_flat

order_items = []
for d in docs:
    if not d["items"] and KEEP_EMPTY_ORDERS:
        order_items.append({"order_id": d["order_id"], "item_idx": None, "sku": None,
                            "qty": None, "unit_price": None})
    for idx, item in enumerate(d["items"]):
        order_items.append({
            "order_id": d["order_id"],
            "item_idx": idx,
            "sku": item["sku"],
            "qty": int(item["qty"]),
            "unit_price": float(item["unit_price"]),
        })

**Choice for empty orders - model answer:** this solution **drops** them (`KEEP_EMPTY_ORDERS = False`, 341 rows): `order_items` holds real items only, so sums and counts are not polluted by NULL rows, and the 40 empty orders are still visible in `orders_flat`. Keeping them (381 rows, like `explode_outer`) is also fine if argued: then a downstream inner join on `order_items` cannot silently lose orders. Both settings pass the self-check.

## Self-checks (do not edit)

In [ ]:
from collections import Counter

# orders_flat
assert len(orders_flat) == 200, f"expected 200 rows (one per document), got {len(orders_flat)}"
print("OK orders_flat has 200 rows")
assert sum(r["email"] is None for r in orders_flat) == 0, "some emails are None - check the v2 location"
print("OK zero null emails")
assert all(isinstance(r["total"], float) or r["total"] is None for r in orders_flat), "total must be float or None"
assert sum(r["total"] is None for r in orders_flat) == 11, "exactly 11 orders have no total"
print("OK total is a number (11 missing)")
assert {r["status"] for r in orders_flat} == {"created", "paid", "shipped", "delivered", "cancelled", None}
assert sum(r["status"] is None for r in orders_flat) == 7
print("OK status is lowercase (7 missing)")
assert all(isinstance(r["created_at"], datetime) and r["created_at"].tzinfo is None for r in orders_flat), \
    "created_at must be a naive UTC datetime for every row"
assert all(r["created_at"].year == 2026 for r in orders_flat), "dates outside 2026 - check the epoch-millis conversion"
print("OK created_at is a datetime for all 200 rows")

# order_items
assert KEEP_EMPTY_ORDERS in (True, False), "set KEEP_EMPTY_ORDERS to True or False"
expected_rows = 381 if KEEP_EMPTY_ORDERS else 341
assert len(order_items) == expected_rows, f"expected {expected_rows} item rows, got {len(order_items)}"
print(f"OK order_items has {expected_rows} rows (KEEP_EMPTY_ORDERS={KEEP_EMPTY_ORDERS})")
real_items = [r for r in order_items if r["item_idx"] is not None]
assert all(type(r["qty"]) is int for r in real_items), "qty must be int"
assert sum(r["qty"] for r in real_items) == 682, "total quantity should be 682"
print("OK qty is int (total quantity 682)")
print("HW2 self-check passed")

**Reflection - model answer:** MERGE on `_id`. It is unique and immutable, and change events carry it as `documentKey` (slides 12, 36); `order_id` has duplicates from the app's retry bug (ORD-10010, ORD-10011).